# SCP01 — Phase 2 EfficientNetB0 (HAM10000)

**GitHub:** [Ashu863804/SCP](https://github.com/Ashu863804/SCP)  
**Kaggle notebook:** SCP01

Business logic is in `src/`. Cell 1 clones the repo, then later cells import from `src`.

## 1. Setup & imports

In [ ]:
# --- 1) Clone GitHub repo (always — no if checks) ---
# Repo: https://github.com/Ashu863804/SCP
!rm -rf /kaggle/working/SCP
!git clone https://github.com/Ashu863804/SCP.git /kaggle/working/SCP

# --- 2) Path bootstrap (must run BEFORE any `import src...`) ---
import os
import sys
from pathlib import Path

GITHUB_USER = "Ashu863804"
REPO_NAME = "SCP"
REPO_DIR = Path(f"/kaggle/working/{REPO_NAME}")

def _bootstrap_project_path() -> Path:
    """Locate folder with src/config.py and add it to sys.path."""
    candidates = [
        REPO_DIR,
        REPO_DIR / "skin-cancer-detection",
        Path("/kaggle/working"),
        Path.cwd(),
        Path.cwd().parent,
        *list(Path.cwd().parents)[:6],
    ]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        candidates.extend(sorted(kaggle_input.iterdir(), key=lambda p: p.name))

    seen = set()
    for root in candidates:
        try:
            root = root.resolve()
        except OSError:
            continue
        if root in seen:
            continue
        seen.add(root)
        if (root / "src" / "config.py").is_file():
            root_str = str(root)
            if root_str not in sys.path:
                sys.path.insert(0, root_str)
            os.chdir(root)
            return root

    raise FileNotFoundError(
        f"src/config.py not found after cloning https://github.com/{GITHUB_USER}/{REPO_NAME}. "
        "Ensure the GitHub repo contains the src/ folder at its root."
    )

PROJECT_ROOT = _bootstrap_project_path()

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__)
print("Project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src" / "config.py").exists())

## 2. Configuration

In [ ]:
from src.config import get_config

config = get_config()

# Uncomment and set if your Kaggle dataset slug differs:
# config.kaggle_dataset_slug = "datasets/kmader/skin-cancer-mnist-ham10000"
# config.__post_init__()

print("Kaggle input:", config.kaggle_input_dir)
print("Organized data:", config.organized_dir)
print("Outputs:", config.output_dir)

## 3. Dataset loading

In [ ]:
from src.dataset import prepare_dataset_pipeline

data = prepare_dataset_pipeline(config, force_organize=False)

train_generator = data["train_generator"]
val_generator = data["val_generator"]
test_generator = data["test_generator"]
class_weight_dict = data["class_weight_dict"]
CLASS_NAMES = data["class_names"]
NUM_CLASSES = data["num_classes"]

print("Classes:", CLASS_NAMES)
print("Class weights:", class_weight_dict)

## 4. Model creation

In [ ]:
from src.model import create_and_compile_model

model, base_model = create_and_compile_model(NUM_CLASSES, config)
model.summary()

## 5. Training (frozen backbone)

In [ ]:
from src.train import get_callbacks, train_model

callbacks = get_callbacks(config)
history = train_model(
    model,
    train_generator,
    val_generator,
    config,
    class_weight_dict=class_weight_dict,
    callbacks=callbacks,
)

## 6. Fine-tuning (top layers — same as original notebook)

In [ ]:
from src.train import fine_tune_model

history_fine = fine_tune_model(
    model,
    base_model,
    train_generator,
    val_generator,
    config,
    class_weight_dict=class_weight_dict,
    callbacks=callbacks,
)

## 7. Evaluation

In [ ]:
from src.evaluate import evaluate_model
from src.utils import save_text_report
from src.train import save_final_model

results = evaluate_model(model, test_generator, CLASS_NAMES)
save_text_report(results["classification_report"], config.classification_report_path)
save_final_model(model, config.final_model_path)
print(f"Test accuracy: {results['accuracy']:.4f}")

## 8. Plots & visualizations

In [ ]:
from src.utils import plot_training_history, plot_confusion_matrix, plot_sample_predictions

plot_training_history(history, save_path=config.history_plot_path)
plot_confusion_matrix(
    results["y_true"],
    results["y_pred"],
    CLASS_NAMES,
    save_path=config.confusion_matrix_path,
)

images, labels = next(test_generator)
preds = model.predict(images)
plot_sample_predictions(images, labels, preds, CLASS_NAMES)